In [1]:
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, round, when, max, min, countDistinct

builder = SparkSession.builder \
    .appName("hospital-gold") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.shuffle.partitions", "8")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

PROJECT_BASE = os.path.expanduser(
    "~/data-engineering-pathway/projects/hospital-lakehouse"
)
DELTA_BASE = f"{PROJECT_BASE}/delta"

# Create Gold directories
for table in ["admission_summary", "hospital_revenue",
              "doctor_performance", "disease_trends", "insurance_analysis"]:
    os.makedirs(f"{DELTA_BASE}/gold/{table}", exist_ok=True)

print(f"Spark OK  : {spark.version}")
print(f"Delta base: {DELTA_BASE}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/26 01:37:37 WARN Utils: Your hostname, M-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.100.4 instead (on interface en0)
26/04/26 01:37:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/malone/data-engineering-pathway/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/malone/.ivy2.5.2/cache
The jars for the packages stored in: /Users/malone/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-79d16ce9-9cc7-4ad6-a9b6-f1ec23b5e501;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j-api;2.0

Spark OK  : 4.1.1
Delta base: /Users/malone/data-engineering-pathway/projects/hospital-lakehouse/delta


In [2]:
# Load Silver tables
silver_patients   = spark.read.format("delta").load(f"{DELTA_BASE}/silver/patients")
silver_admissions = spark.read.format("delta").load(f"{DELTA_BASE}/silver/admissions")

print(f"silver_patients   : {silver_patients.count()} rows")
print(f"silver_admissions : {silver_admissions.count()} rows")

# Join on admission_id — this is the key relationship
joined_df = silver_admissions.join(
    silver_patients,
    on="admission_id",
    how="left"
)

print(f"Joined rows       : {joined_df.count()}")

# Register as temp view for Spark SQL
joined_df.createOrReplaceTempView("hospital_data")
print("Temp view created : hospital_data")
joined_df.show(3, truncate=False)

silver_patients   : 55500 rows
silver_admissions : 55500 rows


Joined rows       : 55500
Temp view created : hospital_data
+------------+-----------------+-----------------+--------------+-----------------------+------------------+--------------+-----------+--------------+--------------+----------+------------+-------------------+----------------+------------+----------------+---+-------------------+------+----------+
|admission_id|medical_condition|date_of_admission|doctor        |hospital               |insurance_provider|billing_amount|room_number|admission_type|discharge_date|medication|test_results|length_of_stay_days|billing_category|data_quality|patient_id      |age|age_group          |gender|blood_type|
+------------+-----------------+-----------------+--------------+-----------------------+------------------+--------------+-----------+--------------+--------------+----------+------------+-------------------+----------------+------------+----------------+---+-------------------+------+----------+
|17179869184 |OBESITY          |2023-05-22 

In [3]:
gold_admission_summary = spark.sql("""
    SELECT
        medical_condition,
        admission_type,
        COUNT(*)                                    AS total_admissions,
        ROUND(AVG(length_of_stay_days), 1)         AS avg_length_of_stay_days,
        MIN(length_of_stay_days)                   AS min_stay_days,
        MAX(length_of_stay_days)                   AS max_stay_days,
        ROUND(AVG(billing_amount), 2)              AS avg_billing_amount,
        COUNT(CASE WHEN test_results = 'ABNORMAL'
            THEN 1 END)                             AS abnormal_results_count,
        ROUND(COUNT(CASE WHEN test_results = 'ABNORMAL' THEN 1 END)
            * 100.0 / COUNT(*), 1)                 AS abnormal_rate_pct
    FROM hospital_data
    GROUP BY medical_condition, admission_type
    ORDER BY total_admissions DESC
""")

gold_admission_summary.write \
    .format("delta").mode("overwrite") \
    .save(f"{DELTA_BASE}/gold/admission_summary")

print(f"Gold Table 1 — Admission Summary: {gold_admission_summary.count()} rows")
gold_admission_summary.show(10, truncate=False)

Gold Table 1 — Admission Summary: 18 rows
+-----------------+--------------+----------------+-----------------------+-------------+-------------+------------------+----------------------+-----------------+
|medical_condition|admission_type|total_admissions|avg_length_of_stay_days|min_stay_days|max_stay_days|avg_billing_amount|abnormal_results_count|abnormal_rate_pct|
+-----------------+--------------+----------------+-----------------------+-------------+-------------+------------------+----------------------+-----------------+
|DIABETES         |URGENT        |3229            |15.1                   |1            |30           |25457.48          |1085                  |33.6             |
|HYPERTENSION     |ELECTIVE      |3221            |15.5                   |1            |30           |25577.05          |1031                  |32.0             |
|CANCER           |ELECTIVE      |3148            |15.3                   |1            |30           |25698.6           |1051            

In [4]:
gold_hospital_revenue = spark.sql("""
    SELECT
        hospital,
        COUNT(*)                                        AS total_admissions,
        ROUND(SUM(billing_amount), 2)                  AS total_revenue,
        ROUND(AVG(billing_amount), 2)                  AS avg_billing_amount,
        COUNT(CASE WHEN billing_category = 'high'
            THEN 1 END)                                 AS high_billing_cases,
        COUNT(CASE WHEN billing_category = 'mid'
            THEN 1 END)                                 AS mid_billing_cases,
        COUNT(CASE WHEN billing_category = 'low'
            THEN 1 END)                                 AS low_billing_cases,
        ROUND(AVG(length_of_stay_days), 1)             AS avg_length_of_stay,
        COUNT(DISTINCT medical_condition)              AS conditions_treated
    FROM hospital_data
    GROUP BY hospital
    ORDER BY total_revenue DESC
    LIMIT 20
""")

gold_hospital_revenue.write \
    .format("delta").mode("overwrite") \
    .save(f"{DELTA_BASE}/gold/hospital_revenue")

print(f"Gold Table 2 — Hospital Revenue: {gold_hospital_revenue.count()} rows")
gold_hospital_revenue.show(10, truncate=False)

Gold Table 2 — Hospital Revenue: 20 rows
+-----------+----------------+-------------+------------------+------------------+-----------------+-----------------+------------------+------------------+
|hospital   |total_admissions|total_revenue|avg_billing_amount|high_billing_cases|mid_billing_cases|low_billing_cases|avg_length_of_stay|conditions_treated|
+-----------+----------------+-------------+------------------+------------------+-----------------+-----------------+------------------+------------------+
|Johnson PLC|38              |1084202.7    |28531.65          |27                |7                |4                |16.3              |6                 |
|LLC Smith  |44              |1030189.88   |23413.41          |20                |20               |4                |15.6              |6                 |
|Smith PLC  |36              |1029424.47   |28595.12          |26                |10               |0                |17.6              |6                 |
|Ltd Smith  |39  

In [5]:
gold_doctor_performance = spark.sql("""
    SELECT
        doctor,
        COUNT(*)                                        AS total_patients,
        COUNT(DISTINCT medical_condition)              AS conditions_treated,
        ROUND(AVG(billing_amount), 2)                  AS avg_billing_amount,
        ROUND(SUM(billing_amount), 2)                  AS total_revenue_generated,
        ROUND(AVG(length_of_stay_days), 1)             AS avg_length_of_stay,
        COUNT(CASE WHEN test_results = 'NORMAL'
            THEN 1 END)                                 AS normal_results,
        COUNT(CASE WHEN test_results = 'ABNORMAL'
            THEN 1 END)                                 AS abnormal_results,
        COUNT(CASE WHEN test_results = 'INCONCLUSIVE'
            THEN 1 END)                                 AS inconclusive_results
    FROM hospital_data
    GROUP BY doctor
    ORDER BY total_patients DESC
    LIMIT 20
""")

gold_doctor_performance.write \
    .format("delta").mode("overwrite") \
    .save(f"{DELTA_BASE}/gold/doctor_performance")

print(f"Gold Table 3 — Doctor Performance: {gold_doctor_performance.count()} rows")
gold_doctor_performance.show(10, truncate=False)

Gold Table 3 — Doctor Performance: 20 rows
+-----------------+--------------+------------------+------------------+-----------------------+------------------+--------------+----------------+--------------------+
|doctor           |total_patients|conditions_treated|avg_billing_amount|total_revenue_generated|avg_length_of_stay|normal_results|abnormal_results|inconclusive_results|
+-----------------+--------------+------------------+------------------+-----------------------+------------------+--------------+----------------+--------------------+
|Michael Smith    |27            |6                 |29055.62          |784501.82              |15.6              |6             |12              |9                   |
|Robert Smith     |22            |6                 |28854.23          |634792.99              |13.7              |7             |6               |9                   |
|John Smith       |22            |6                 |27732.26          |610109.62              |14.4            

In [6]:
gold_disease_trends = spark.sql("""
    SELECT
        medical_condition,
        age_group,
        COUNT(*)                                        AS total_cases,
        ROUND(AVG(length_of_stay_days), 1)             AS avg_stay_days,
        ROUND(AVG(billing_amount), 2)                  AS avg_billing,
        COUNT(CASE WHEN admission_type = 'EMERGENCY'
            THEN 1 END)                                 AS emergency_admissions,
        ROUND(COUNT(CASE WHEN admission_type = 'EMERGENCY' THEN 1 END)
            * 100.0 / COUNT(*), 1)                     AS emergency_rate_pct,
        COUNT(CASE WHEN test_results = 'ABNORMAL'
            THEN 1 END)                                 AS abnormal_tests
    FROM hospital_data
    GROUP BY medical_condition, age_group
    ORDER BY medical_condition, total_cases DESC
""")

gold_disease_trends.write \
    .format("delta").mode("overwrite") \
    .save(f"{DELTA_BASE}/gold/disease_trends")

print(f"Gold Table 4 — Disease Trends: {gold_disease_trends.count()} rows")
gold_disease_trends.show(15, truncate=False)

Gold Table 4 — Disease Trends: 24 rows
+-----------------+-------------------+-----------+-------------+-----------+--------------------+------------------+--------------+
|medical_condition|age_group          |total_cases|avg_stay_days|avg_billing|emergency_admissions|emergency_rate_pct|abnormal_tests|
+-----------------+-------------------+-----------+-------------+-----------+--------------------+------------------+--------------+
|ARTHRITIS        |Adult (36-60)      |3485       |15.4         |25383.44   |1157                |33.2              |1236          |
|ARTHRITIS        |Senior (61+)       |3394       |15.6         |25762.74   |1143                |33.7              |1140          |
|ARTHRITIS        |Young Adult (18-35)|2408       |15.5         |25258.55   |798                 |33.1              |802           |
|ARTHRITIS        |Paediatric (0-17)  |21         |15.9         |28881.38   |10                  |47.6              |10            |
|ASTHMA           |Senior (61+

In [7]:
gold_insurance_analysis = spark.sql("""
    SELECT
        insurance_provider,
        COUNT(*)                                        AS total_claims,
        ROUND(SUM(billing_amount), 2)                  AS total_billed,
        ROUND(AVG(billing_amount), 2)                  AS avg_claim_amount,
        COUNT(CASE WHEN billing_category = 'high'
            THEN 1 END)                                 AS high_cost_claims,
        ROUND(COUNT(CASE WHEN billing_category = 'high' THEN 1 END)
            * 100.0 / COUNT(*), 1)                     AS high_cost_rate_pct,
        COUNT(DISTINCT medical_condition)              AS conditions_covered,
        ROUND(AVG(length_of_stay_days), 1)             AS avg_covered_stay_days
    FROM hospital_data
    GROUP BY insurance_provider
    ORDER BY total_billed DESC
""")

gold_insurance_analysis.write \
    .format("delta").mode("overwrite") \
    .save(f"{DELTA_BASE}/gold/insurance_analysis")

print(f"Gold Table 5 — Insurance Analysis: {gold_insurance_analysis.count()} rows")
gold_insurance_analysis.show(truncate=False)

Gold Table 5 — Insurance Analysis: 5 rows
+------------------+------------+--------------+----------------+----------------+------------------+------------------+---------------------+
|insurance_provider|total_claims|total_billed  |avg_claim_amount|high_cost_claims|high_cost_rate_pct|conditions_covered|avg_covered_stay_days|
+------------------+------------+--------------+----------------+----------------+------------------+------------------+---------------------+
|Cigna             |11249       |2.8713934516E8|25525.77        |6896            |61.3              |6                 |15.5                 |
|Medicare          |11154       |2.857207577E8 |25615.99        |6872            |61.6              |6                 |15.6                 |
|Blue Cross        |11059       |2.8325429421E8|25613.01        |6814            |61.6              |6                 |15.5                 |
|UnitedHealthcare  |11125       |2.8245454259E8|25389.17        |6751            |60.7              

In [8]:
tables = [
    ("admission_summary",   "Admission Summary"),
    ("hospital_revenue",    "Hospital Revenue"),
    ("doctor_performance",  "Doctor Performance"),
    ("disease_trends",      "Disease Trends"),
    ("insurance_analysis",  "Insurance Analysis"),
]

print("=" * 60)
print("GOLD LAYER — HOSPITAL COMPLETE")
print("=" * 60)

for folder, label in tables:
    df = spark.read.format("delta").load(f"{DELTA_BASE}/gold/{folder}")
    print(f"  {label:<25} → {df.count()} rows")

print("=" * 60)
print("Pipeline: Bronze → Silver → Gold complete")
print("Dataset : 55,500 patient admissions")
print("PII     : Masked at Silver, absent from Gold")
print("=" * 60)

GOLD LAYER — HOSPITAL COMPLETE
  Admission Summary         → 18 rows
  Hospital Revenue          → 20 rows
  Doctor Performance        → 20 rows
  Disease Trends            → 24 rows
  Insurance Analysis        → 5 rows
Pipeline: Bronze → Silver → Gold complete
Dataset : 55,500 patient admissions
PII     : Masked at Silver, absent from Gold
